# Follow-up Experiments — Interactive Kaggle & VS Code Runner

Run this notebook **in the Kaggle session attached to this notebook in VS Code** (or directly on Kaggle web).

### Your Datasets on Kaggle:
- **Structured3D**: `lokeshmotwani213/structured3d-panoramic-depth`
- **PanoX**: `lokeshmotwani213/panox-panoramic-depth`
- **Project Code**: `lokeshmotwani213/lokesh-code`

In the Kaggle sidebar under **Input > + Add Input**, search for and attach these datasets.

The setup cell will:
1. Use your attached `lokesh-code` (or existing `/kaggle/working/Lokesh`);
2. Automatically detect and extract the attached PanoX and Structured3D datasets;
3. Download model code/weights (`DA360_large.pth`, `dpt_hybrid_384.pt`);
4. Stream experiment execution in real-time to this VS Code window.

In [5]:
import os
import sys
import subprocess
from pathlib import Path

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

gpu_status = subprocess.run(["nvidia-smi"], check=False)
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError(
        "This notebook is attached to a CPU kernel. In Kaggle, select "
        "Settings > Accelerator > GPU T4 x2, then restart and run from the top."
    )
if gpu_status.returncode != 0:
    print("Warning: nvidia-smi returned a non-zero exit code.")

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Wed Nov  4 20:39:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                           

In [6]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path
from typing import Optional

import torch

# ---------------------------------------------------------------------
# A. Select the experiment BEFORE running this setup cell.
# ---------------------------------------------------------------------
RUN = "01"                 # "01", "02", "03", "04", "05", or "all"
LIMIT = 16                 # Pilot size for 01/04/05; set None for full runs
RUN_FINETUNE_IN_ALL = False

# Leave dataset roots as None for automatic Kaggle Input discovery.
DATA_PATHS = {
    "PANOX_ROOT": None,
    "STRUCTURED3D_ROOT": None,
    "DA360_REPO": "/kaggle/working/DA360",
    "DA360_WEIGHTS": "/kaggle/working/da360_weights/DA360_large.pth",
    "MIDAS_WEIGHTS": "/kaggle/working/midas_weights/dpt_hybrid_384.pt",
}

# Optional GitHub PAT if cloning private repo:
GITHUB_PAT = os.environ.get("GITHUB_PAT", "")
REPO_URL = (
    "https://github.com/AurindumBanerjee/"
    "Distortion-Aware-Panoramic-Depth-Estimation.git"
)
WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

if RUN not in {"01", "02", "03", "04", "05", "all"}:
    raise ValueError(f"Unknown RUN={RUN!r}")
if not WORKING.is_dir():
    raise RuntimeError("This does not look like a Kaggle runtime: /kaggle/working is missing.")


# ---------------------------------------------------------------------
# Helper: sanitize all shell scripts and python files
# ---------------------------------------------------------------------
def _sanitize_scripts(root: Path) -> int:
    """Ensure Linux LF line endings, fix escaped bash variable syntax, and chmod +x."""
    fixed = 0
    for sh in root.rglob("*.sh"):
        try:
            raw = sh.read_bytes()
            clean = (
                raw.replace(b"\r\n", b"\n")
                .replace(b"\r", b"\n")
                .replace(b"\\${", b"${")
            )
            if clean != raw:
                sh.write_bytes(clean)
                fixed += 1
            sh.chmod(0o755)
        except Exception:
            pass
    for py in root.rglob("*.py"):
        try:
            raw = py.read_bytes()
            clean = raw.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
            if clean != raw:
                py.write_bytes(clean)
                fixed += 1
        except Exception:
            pass
    return fixed


# ---------------------------------------------------------------------
# B. Locate or bootstrap the project into writable /kaggle/working.
# ---------------------------------------------------------------------
def _is_lokesh_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "run_experiment.py").is_file()
        and (path / "followup_experiments" / "run_all.sh").is_file()
    )


def _is_writable_directory(path: Path) -> bool:
    try:
        probe = path / ".kaggle_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except OSError:
        return False


def _git_output(args, cwd=None):
    return subprocess.run(
        args, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, check=False
    )


def _verify_or_update_experiment_revision(lokesh_root: Path) -> None:
    top = _git_output(
        ["git", "-C", str(lokesh_root), "rev-parse", "--show-toplevel"]
    )
    if top.returncode != 0:
        print("Project source has no Git metadata; using files directly from workspace.")
        return
    print("Project Git repository found at:", top.stdout.strip())


candidate_roots = []
if os.environ.get("LOKESH_ROOT"):
    candidate_roots.append(Path(os.environ["LOKESH_ROOT"]).expanduser())

# Common writable and working locations
candidate_roots.extend([
    Path.cwd(),
    Path.cwd() / "Lokesh",
    WORKING / "Lokesh",
    WORKING / "Distortion-Aware-Panoramic-Depth-Estimation" / "Lokesh",
])

# Deep recursive search across all /kaggle/input mounts
if INPUT.is_dir():
    print("Searching for Lokesh code under /kaggle/input...")
    # 1. Look for run_experiment.py anywhere in /kaggle/input
    for run_file in INPUT.rglob("run_experiment.py"):
        folder = run_file.parent
        if _is_lokesh_root(folder):
            print(f"Found code source in input: {folder}")
            candidate_roots.append(folder)

    # 2. Check for attached code archives (.zip)
    for z in INPUT.rglob("*.zip"):
        if any(k in z.name.lower() for k in ("code", "lokesh", "panoramic")):
            dest = WORKING / "Lokesh"
            if not dest.exists():
                print(f"Unpacking attached code archive {z} to {dest}...")
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(WORKING)
            if _is_lokesh_root(dest):
                candidate_roots.append(dest)

# 1. First check if a writable Lokesh root already exists
LOKESH_ROOT = next(
    (p.resolve() for p in candidate_roots if _is_lokesh_root(p) and _is_writable_directory(p)),
    None,
)

# 2. If only found in read-only /kaggle/input, copy into writable /kaggle/working/Lokesh
input_source = next(
    (
        p.resolve() for p in candidate_roots
        if _is_lokesh_root(p)
        and p.resolve().is_relative_to(INPUT.resolve())
    ),
    None,
)

if LOKESH_ROOT is None:
    if input_source is not None:
        destination = WORKING / "Lokesh"
        if destination.exists() and not _is_lokesh_root(destination):
            shutil.rmtree(destination)
        if not destination.exists():
            def _ignore_non_code(_directory, names):
                return {
                    name for name in names
                    if name in {".git", "__pycache__", ".pytest_cache", "outputs", "logs", "weights", "data"}
                }
            print(f"Copying project from read-only mount {input_source} to {destination}...")
            shutil.copytree(input_source, destination, ignore=_ignore_non_code)
        LOKESH_ROOT = destination.resolve()

# 3. If still not found, try git clone
if LOKESH_ROOT is None:
    repo_dir = WORKING / "Distortion-Aware-Panoramic-Depth-Estimation"
    if not repo_dir.exists():
        print("No project found in /kaggle/input. Trying git clone...")
        clone_url = REPO_URL
        if GITHUB_PAT:
            clone_url = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
        try:
            subprocess.check_call([
                "git", "clone", "--branch", "main", "--depth", "1", clone_url, str(repo_dir)
            ])
        except Exception as e:
            print("Git clone failed:", e)
    candidate = repo_dir / "Lokesh"
    if _is_lokesh_root(candidate):
        LOKESH_ROOT = candidate.resolve()

if LOKESH_ROOT is None:
    raise RuntimeError(
        "Could not find Lokesh project folder. "
        "Visible in /kaggle/input: " + str([str(p.relative_to(INPUT)) for p in INPUT.rglob('*') if p.is_dir()][:10])
    )

# Sanitize scripts and python files
sanitized = _sanitize_scripts(LOKESH_ROOT)
if sanitized > 0:
    print(f"Sanitized line endings / permissions on {sanitized} files in {LOKESH_ROOT}")

_verify_or_update_experiment_revision(LOKESH_ROOT)
os.environ["LOKESH_ROOT"] = str(LOKESH_ROOT)
os.chdir(LOKESH_ROOT)
if str(LOKESH_ROOT) not in sys.path:
    sys.path.insert(0, str(LOKESH_ROOT))
shared_dir = LOKESH_ROOT / "followup_experiments" / "shared"
if shared_dir.is_dir() and str(shared_dir) not in sys.path:
    sys.path.insert(0, str(shared_dir))
print("Using writable Lokesh project:", LOKESH_ROOT)

# ---------------------------------------------------------------------
# C. Check GPUs and install helper packages.
# ---------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select GPU T4 x2 in the Kaggle session settings.")
gpu_count = torch.cuda.device_count()
print("CUDA devices:", gpu_count)
for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if (RUN == "03" or (RUN == "all" and RUN_FINETUNE_IN_ALL)) and gpu_count < 2:
    raise RuntimeError("Experiment 03 requires both T4 GPUs (GPU T4 x2).")
subprocess.run(["nvidia-smi"], check=False)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm==0.6.13", "einops", "huggingface_hub", "OpenEXR>=3.2"
])

# ---------------------------------------------------------------------
# D. Discover attached datasets or extract an attached ZIP into /kaggle/working.
# ---------------------------------------------------------------------
def _classify_test_file(test_file: Path) -> Optional[str]:
    path_text = str(test_file).lower()
    if "panox" in path_text:
        return "PANOX"
    if "structured3d" in path_text or "structured_3d" in path_text:
        return "STRUCTURED3D"
    try:
        sample = test_file.read_text(encoding="utf-8", errors="ignore")[:1_000_000].lower()
    except OSError:
        return None
    if ".exr" in sample:
        return "PANOX"
    if "scene_" in sample:
        return "STRUCTURED3D"
    return None


def _classify_zip(archive: Path) -> Optional[str]:
    path_text = str(archive).lower()
    if "panox" in path_text:
        return "PANOX"
    if "structured3d" in path_text or "structured_3d" in path_text:
        return "STRUCTURED3D"
    try:
        with zipfile.ZipFile(archive) as zf:
            names = zf.namelist()
            if any(name.lower().endswith(".exr") for name in names):
                return "PANOX"
            test_names = [name for name in names if Path(name).name.lower() == "test.txt"]
            for name in test_names[:3]:
                raw = zf.read(name)[:1_000_000].decode("utf-8", errors="ignore").lower()
                if ".exr" in raw:
                    return "PANOX"
                if "scene_" in raw:
                    return "STRUCTURED3D"
    except (OSError, zipfile.BadZipFile, KeyError):
        pass
    return None


def _unique_path(paths, what: str) -> Path:
    unique = sorted({p.resolve() for p in paths})
    if len(unique) == 1:
        return unique[0]
    if not unique:
        raise FileNotFoundError(f"No {what} found.")
    rendered = "\n".join(f"  - {p}" for p in unique)
    raise RuntimeError(
        f"Found multiple possible {what} locations. Set the corresponding DATA_PATHS "
        f"entry to the exact folder or ZIP:\n{rendered}"
    )


def _find_test_root(search_root: Path, dataset_kind: str) -> Optional[Path]:
    if not search_root.exists() or not search_root.is_dir():
        return None
    tests = list(search_root.rglob("test.txt"))
    matching = [p.parent for p in tests if _classify_test_file(p) == dataset_kind]
    if not matching and (search_root / "test.txt").is_file():
        matching = [search_root]
    if len(set(matching)) == 1:
        return matching[0].resolve()
    if len(set(matching)) > 1:
        return _unique_path(matching, f"{dataset_kind} test.txt roots")
    return None


def _extract_archive(archive: Path, dataset_kind: str) -> Path:
    destination = WORKING / "followup_data" / dataset_kind.lower()
    existing = _find_test_root(destination, dataset_kind)
    if existing is not None:
        print("Reusing extracted dataset:", existing)
        return existing

    with zipfile.ZipFile(archive) as zf:
        total_uncompressed = sum(info.file_size for info in zf.infolist())
        free_bytes = shutil.disk_usage(WORKING).free
        if total_uncompressed > free_bytes * 0.90:
            raise RuntimeError(
                f"{archive.name} needs about {total_uncompressed / (1024**3):.1f} GB "
                f"when extracted, but only {free_bytes / (1024**3):.1f} GB is free in "
                "/kaggle/working. Attach the extracted dataset or free working storage."
            )
        destination.mkdir(parents=True, exist_ok=True)
        print(
            f"Extracting {archive.name} ({total_uncompressed / (1024**3):.1f} GB uncompressed) "
            f"to {destination}..."
        )
        zf.extractall(destination)

    resolved = _find_test_root(destination, dataset_kind)
    if resolved is None:
        raise RuntimeError(
            f"Extracted {archive.name} into {destination}, but could not find a {dataset_kind} "
            "test.txt folder inside it."
        )
    return resolved


def resolve_dataset_root(env_key: str, dataset_kind: str) -> Path:
    configured = DATA_PATHS.get(env_key)
    if configured:
        p = Path(configured).expanduser()
        if not p.is_absolute():
            p = (LOKESH_ROOT / p).resolve()
        if p.is_file() and p.suffix.lower() == ".zip":
            resolved = _extract_archive(p, dataset_kind)
        elif p.is_dir():
            resolved = _find_test_root(p, dataset_kind) or p
        else:
            raise FileNotFoundError(f"Configured {env_key}={p} not found.")
    else:
        # Search anywhere under /kaggle/input
        test_roots = [
            p.parent for p in INPUT.rglob("test.txt")
            if _classify_test_file(p) == dataset_kind
        ]
        archives = [
            p for p in INPUT.rglob("*.zip")
            if _classify_zip(p) == dataset_kind
        ]
        if test_roots:
            resolved = _unique_path(test_roots, f"{dataset_kind} dataset roots")
        elif len(archives) == 1:
            resolved = _extract_archive(archives[0], dataset_kind)
        elif len(archives) > 1:
            raise RuntimeError(
                f"Multiple {dataset_kind} ZIPs found: {archives}. Set {env_key} explicitly."
            )
        else:
            raise FileNotFoundError(
                f"Could not auto-detect {dataset_kind}. Attach its Kaggle Dataset in the Input "
                f"panel, or set DATA_PATHS['{env_key}'] to the folder/ZIP. "
                f"Visible inputs: {[p.name for p in INPUT.iterdir()]}"
            )

    os.environ[env_key] = str(resolved.resolve())
    print(f"{env_key} => {resolved}  [test.txt found]")
    return resolved


needs_da360 = RUN in ("01", "02", "03", "all")
needs_midas = RUN in ("04", "05", "all")
needs_panox = RUN in ("02", "03", "04", "05", "all")
needs_structured = RUN in ("01", "all")

if needs_panox:
    panox_root = resolve_dataset_root("PANOX_ROOT", "PANOX")
if needs_structured:
    structured_root = resolve_dataset_root("STRUCTURED3D_ROOT", "STRUCTURED3D")

# ---------------------------------------------------------------------
# E. Download only the model source/checkpoint required by this run.
# ---------------------------------------------------------------------
if needs_da360:
    da360_repo = Path(DATA_PATHS["DA360_REPO"])
    if not (da360_repo / "checkpoint.py").is_file() or not (da360_repo / "networks").is_dir():
        if da360_repo.exists():
            raise RuntimeError(
                f"{da360_repo} exists but is not the expected official DA360 checkout "
                "(checkpoint.py and networks/ are required). Remove/rename that folder or "
                "set DATA_PATHS['DA360_REPO'] to a clean path."
            )
        da360_repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/antigravity-tech/DA360.git", str(da360_repo)
        ])

    da360_weight = Path(DATA_PATHS["DA360_WEIGHTS"])
    da360_weight.parent.mkdir(parents=True, exist_ok=True)
    if not da360_weight.is_file() or da360_weight.stat().st_size < 1_000_000:
        from huggingface_hub import hf_hub_download
        print("Downloading the DA360 Large checkpoint (about 1.34 GB)...")
        downloaded = hf_hub_download(
            repo_id="hljiang/DA360",
            filename="DA360_large.pth",
            local_dir=str(da360_weight.parent),
        )
        da360_weight = Path(downloaded)
    os.environ["DA360_REPO"] = str(da360_repo.resolve())
    os.environ["DA360_WEIGHTS"] = str(da360_weight.resolve())
    print("DA360 repo:", da360_repo, "| checkpoint:", da360_weight)

if needs_midas:
    midas_weight = Path(DATA_PATHS["MIDAS_WEIGHTS"])
    midas_weight.parent.mkdir(parents=True, exist_ok=True)
    if not midas_weight.is_file() or midas_weight.stat().st_size < 100_000_000:
        midas_url = "https://github.com/isl-org/MiDaS/releases/download/v3/dpt_hybrid_384.pt"
        print("Downloading MiDaS DPT-Hybrid 384 checkpoint (about 470 MB)...")
        try:
            subprocess.check_call(["wget", "-q", "-O", str(midas_weight), midas_url])
        except Exception:
            import urllib.request
            req = urllib.request.Request(midas_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as resp, open(midas_weight, "wb") as out_f:
                shutil.copyfileobj(resp, out_f)
    os.environ["MIDAS_WEIGHTS"] = str(midas_weight.resolve())
    print("MiDaS checkpoint:", midas_weight)

# ---------------------------------------------------------------------
# F. Check EXR support for PanoX and summarize the runtime configuration.
# ---------------------------------------------------------------------
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
if needs_panox:
    import cv2
    first_line = next(
        (line.split() for line in (panox_root / "test.txt").read_text().splitlines() if line.strip()),
        None,
    )
    if first_line and len(first_line) > 1:
        exr_path = panox_root / first_line[1]
        if exr_path.is_file():
            probe = cv2.imread(
                str(exr_path), cv2.IMREAD_ANYDEPTH | cv2.IMREAD_UNCHANGED
            )
            if probe is None:
                import OpenEXR  # noqa: F401
                print("OpenCV cannot decode the sample EXR; the OpenEXR fallback is available.")
            else:
                print("OpenCV EXR probe succeeded:", exr_path)
        else:
            print("EXR sample path not found from the first list row; loader will validate paths during the run.")

print("\nSetup complete.")
print("Experiment:", RUN, "| pilot LIMIT:", LIMIT)
print("Project root:", LOKESH_ROOT)
print("Outputs will be written under:", LOKESH_ROOT / "followup_experiments")



Searching for Lokesh code under /kaggle/input...
Found code source in input: /kaggle/input/datasets/lokeshmotwani213/lokesh-code/Lokesh
Copying project from read-only mount /kaggle/input/datasets/lokeshmotwani213/lokesh-code/Lokesh to /kaggle/working/Lokesh...
Project source has no Git metadata; using files directly from workspace.
Using writable Lokesh project: /kaggle/working/Lokesh
CUDA devices: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Wed Nov  4 20:39:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |    

Cloning into '/kaggle/working/DA360'...


DA360_large.pth: reconstructing file:   0%|          |  0.00B / 1.34GB            

DA360_large.pth: downloading bytes:           |  0.00B            

DA360 repo: /kaggle/working/DA360 | checkpoint: /kaggle/working/da360_weights/DA360_large.pth

Setup complete.
Experiment: 01 | pilot LIMIT: 16
Project root: /kaggle/working/Lokesh
Outputs will be written under: /kaggle/working/Lokesh/followup_experiments


## Launch the selected experiment

The setup cell above prepares the **same kernel**. Run this launch cell only after setup completes successfully.

Suggested sequence:
- **01:** start with LIMIT=16, then set LIMIT=None for the full Structured3D run.
- **02:** leave all PanoX environments in the list; do not apply a frame limit.
- **04 and 05:** start with LIMIT=2 because these are inference-heavy.
- **03:** trains the optional student and requires both T4 GPUs.
- **all:** runs 01, 02, 04 and 05 sequentially; set RUN_FINETUNE_IN_ALL=True to include 03.

In [7]:
# Run this only after the setup cell has completed successfully.
import os
import sys
import subprocess
from pathlib import Path

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYTHON"] = sys.executable
env["CUDA_VISIBLE_DEVICES"] = (
    "0,1" if RUN == "03" or (RUN == "all" and RUN_FINETUNE_IN_ALL) else "0"
)
env["RUN_FINETUNE"] = "1" if RUN == "all" and RUN_FINETUNE_IN_ALL else "0"

if "LOKESH_ROOT" not in globals() or not LOKESH_ROOT.is_dir():
    raise RuntimeError("LOKESH_ROOT is not set. Run the setup cell first.")

# Sanitize all scripts in LOKESH_ROOT immediately before execution
def _sanitize_scripts(root: Path) -> int:
    fixed = 0
    for sh in root.rglob("*.sh"):
        try:
            raw = sh.read_bytes()
            clean = (
                raw.replace(b"\r\n", b"\n")
                .replace(b"\r", b"\n")
                .replace(b"\\${", b"${")
            )
            if clean != raw:
                sh.write_bytes(clean)
                fixed += 1
            sh.chmod(0o755)
        except Exception:
            pass
    for py in root.rglob("*.py"):
        try:
            raw = py.read_bytes()
            clean = raw.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
            if clean != raw:
                py.write_bytes(clean)
                fixed += 1
        except Exception:
            pass
    return fixed

_sanitized = _sanitize_scripts(LOKESH_ROOT)
if _sanitized > 0:
    print(f"Sanitized {_sanitized} files before launch.")

if RUN == "all":
    cmd = ["bash", "followup_experiments/run_all.sh"]
else:
    folders = {
        "01": "01_scale_only_protocol",
        "02": "02_domain_conditioned_gate",
        "03": "03_panda_style_finetuning",
        "04": "04_mono360_20view",
        "05": "05_fov_resolution_sweep",
    }
    cmd = ["bash", f"followup_experiments/{folders[RUN]}/run_gpu.sh"]
    if LIMIT is not None and RUN in ("01", "04", "05"):
        cmd += ["--limit", str(LIMIT)]

print("Working directory:", LOKESH_ROOT)
print("Command:", " ".join(cmd))
print("CUDA_VISIBLE_DEVICES:", env["CUDA_VISIBLE_DEVICES"])
print("-" * 70)

# Stream live output line-by-line so VS Code displays progress in real-time
proc = subprocess.Popen(
    cmd,
    cwd=LOKESH_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)

proc.stdout.close()
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print("\n" + "=" * 70)
print(f"Experiment {RUN} completed successfully!")



## Student evaluation (after Experiment 03)

After training finishes, copy the printed best.pt path into the cell below. Use an absolute path or a path relative to LOKESH_ROOT. Evaluation uses the held-out environment groups saved with the training run.

In [ ]:
# Student evaluation (after Experiment 03)
# Set CHECKPOINT to the path of best.pt (or leave empty to auto-detect the latest best.pt)
CHECKPOINT = ""

if not CHECKPOINT:
    ckpts = sorted(
        (LOKESH_ROOT / "followup_experiments" / "03_panda_style_finetuning" / "outputs").glob("run_*/best.pt"),
        key=lambda p: p.stat().st_mtime,
    )
    if ckpts:
        CHECKPOINT = str(ckpts[-1])
        print("Auto-detected latest student checkpoint:", CHECKPOINT)

if CHECKPOINT:
    checkpoint_path = Path(CHECKPOINT).expanduser()
    if not checkpoint_path.is_absolute():
        checkpoint_path = (LOKESH_ROOT / checkpoint_path).resolve()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Student checkpoint not found: {checkpoint_path}")

    eval_env = os.environ.copy()
    eval_env["PYTHONUNBUFFERED"] = "1"
    eval_env["CUDA_VISIBLE_DEVICES"] = "0"
    eval_env["PYTHON"] = sys.executable

    cmd = [
        "bash",
        "followup_experiments/03_panda_style_finetuning/evaluate_gpu.sh",
        str(checkpoint_path),
    ]
    print("Evaluating student:", checkpoint_path)
    print("-" * 70)
    proc = subprocess.Popen(
        cmd,
        cwd=LOKESH_ROOT,
        env=eval_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    print("\nStudent evaluation completed successfully!")
else:
    print("No student checkpoint found yet. Run Experiment 03 first.")



## Results and session notes

Every run creates a timestamped folder under:

**Lokesh/followup_experiments/<study>/outputs/run_<timestamp>/**

The main files are REPORT.md, summary.json, per_scene.csv and the run log. They are stored under /kaggle/working because the project checkout is bootstrapped there. Download or save the result folders before ending the Kaggle session.

The setup cell does not start a training/evaluation job automatically: you choose the study, run setup, then execute the launch cell. For Experiment 03, run the student-evaluation cell separately after training.

In [ ]:
# Results Dashboard — Inspect reports and metrics directly in VS Code
import json
from pathlib import Path

all_runs = sorted(
    (LOKESH_ROOT / "followup_experiments").glob("*/outputs/run_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not all_runs:
    print("No run outputs found yet. Execute Cell 4 to run an experiment.")
else:
    print(f"Found {len(all_runs)} experiment run(s):")
    for idx, r in enumerate(all_runs[:5]):
        exp_name = r.parent.parent.name
        print(f"  [{idx}] {exp_name} -> {r.name}")
    print()

    # Display the most recent run
    latest = all_runs[0]
    exp_name = latest.parent.parent.name
    print(f"=== Showing Results for: {exp_name} ({latest.name}) ===")

    report_file = latest / "REPORT.md"
    if report_file.is_file():
        try:
            from IPython.display import display, Markdown
            display(Markdown(report_file.read_text(encoding="utf-8")))
        except Exception:
            print(report_file.read_text(encoding="utf-8"))

    summary_file = latest / "summary.json"
    if summary_file.is_file():
        try:
            summary = json.loads(summary_file.read_text(encoding="utf-8"))
            print("\nSummary Metrics:")
            if "summary" in summary:
                print(json.dumps(summary["summary"], indent=2))
            elif "paired_cluster_bootstrap_scale_vs_affine" in summary:
                print(json.dumps(summary["paired_cluster_bootstrap_scale_vs_affine"], indent=2))
            else:
                print(json.dumps(summary, indent=2)[:2000])
        except Exception as e:
            print("Could not parse summary.json:", e)

